# Analysis 01 direction breakdown

Purpose: inspect the named geometry, dataset, or model diagnostic.

Prerequisites: install the project with the notebook extra and supply the research artifacts selected in the configuration cells. Launch Jupyter from the project root. See `docs/notebooks.md` for per-notebook inputs.

Outputs: displayed diagnostics and, where configured, exported figures/tables. Run cells from top to bottom. Saved outputs have been cleared.


# Analysis 01: Direction Breakdown

Break prediction errors down by offshore and target/predicted direction sectors, with per-site direction bias summaries and polar-style sector diagnostics.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from notebooks.multisource_notebook_helpers import (
    resolve_point_centric_dir,
    results_dir_from_config,
    load_prediction_table,
)

try:
    import contextily as ctx
except Exception:
    ctx = None

try:
    from pyproj import Transformer
except Exception:
    Transformer = None

plt.style.use("seaborn-v0_8-whitegrid")

In [ ]:
CONFIG_PATH = Path("../configs/training.yaml")
RESULTS_DIR_OVERRIDE = Path(
    "../results/FINAL_RESULTS_V2/18_cnn5chan_v1/"
)  # e.g. Path('../results/my_run')
RESULTS_DIR = (
    Path(RESULTS_DIR_OVERRIDE).expanduser()
    if RESULTS_DIR_OVERRIDE
    else results_dir_from_config(CONFIG_PATH)
)
SPLIT = "val"
POINT_CENTRIC_DIR = resolve_point_centric_dir(CONFIG_PATH)
ANALYSIS_OUTPUT_DIR = RESULTS_DIR / "analysis_01_direction_breakdown"
ANALYSIS_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
pred_df = load_prediction_table(RESULTS_DIR, split=SPLIT)
pred_df["target_dir_deg"] = (
    np.degrees(np.arctan2(pred_df["target_dir_sin"], pred_df["target_dir_cos"])) % 360.0
)
pred_df["pred_dir_deg"] = (
    np.degrees(np.arctan2(pred_df["pred_dir_sin"], pred_df["pred_dir_cos"])) % 360.0
)
pred_df["target_dp_deg"] = (
    np.degrees(np.arctan2(pred_df["target_dp_sin"], pred_df["target_dp_cos"])) % 360.0
)
pred_df["pred_dp_deg"] = (
    np.degrees(np.arctan2(pred_df["pred_dp_sin"], pred_df["pred_dp_cos"])) % 360.0
)
pred_df["dir_error_deg"] = (
    (pred_df["pred_dir_deg"] - pred_df["target_dir_deg"] + 180.0) % 360.0
) - 180.0
pred_df["dp_error_deg"] = (
    (pred_df["pred_dp_deg"] - pred_df["target_dp_deg"] + 180.0) % 360.0
) - 180.0
bins = np.arange(0.0, 361.0, 45.0)
pred_df["target_dir_sector"] = pd.cut(
    pred_df["target_dir_deg"], bins=bins, include_lowest=True, right=False
)
pred_df["pred_dir_sector"] = pd.cut(
    pred_df["pred_dir_deg"], bins=bins, include_lowest=True, right=False
)
pred_df["target_dp_sector"] = pd.cut(
    pred_df["target_dp_deg"], bins=bins, include_lowest=True, right=False
)
pred_df.head()

In [ ]:
def summarize_sector_errors(df, sector_col):
    return (
        df.groupby(sector_col, observed=False)
        .agg(
            count=("site", "size"),
            dir_rmse=("dir_error_deg", lambda s: float(np.sqrt(np.nanmean(np.square(s))))),
            dir_bias=("dir_error_deg", "mean"),
            dp_rmse=("dp_error_deg", lambda s: float(np.sqrt(np.nanmean(np.square(s))))),
            dp_bias=("dp_error_deg", "mean"),
        )
        .reset_index()
    )


dp_sector_summary = summarize_sector_errors(pred_df, "target_dp_sector")
dir_sector_summary = summarize_sector_errors(pred_df, "target_dir_sector")
dp_sector_summary

In [ ]:
site_bias = (
    pred_df.groupby("site")
    .agg(
        dir_bias=("dir_error_deg", "mean"),
        dir_rmse=("dir_error_deg", lambda s: float(np.sqrt(np.nanmean(np.square(s))))),
        dp_bias=("dp_error_deg", "mean"),
        dp_rmse=("dp_error_deg", lambda s: float(np.sqrt(np.nanmean(np.square(s))))),
        count=("site", "size"),
    )
    .sort_values("dir_rmse", ascending=False)
)
site_bias.head(20)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].bar(dp_sector_summary["target_dp_sector"].astype(str), dp_sector_summary["dp_rmse"])
axes[0].tick_params(axis="x", rotation=45)
axes[0].set_title("Dp RMSE by target direction sector")
axes[0].set_xlabel("Target Dp sector (deg)")
axes[0].set_ylabel("RMSE (deg)")
axes[1].scatter(pred_df["target_dp_deg"], pred_df["dp_error_deg"], s=4, alpha=0.25)
axes[1].set_title("Direction error polar proxy")
axes[1].set_xlabel("Target Dp (deg)")
axes[1].set_ylabel("Dp error (deg)")
plt.tight_layout()
plt.show()

fig_hist, ax_hist = plt.subplots(figsize=(7, 5))
ax_hist.bar(dp_sector_summary["target_dp_sector"].astype(str), dp_sector_summary["dp_rmse"])
ax_hist.tick_params(axis="x", rotation=45)
ax_hist.set_title("Dp RMSE by target direction sector")
ax_hist.set_xlabel("Target Dp sector (deg)")
ax_hist.set_ylabel("RMSE (deg)")
fig_hist.tight_layout()
fig_hist.savefig(
    ANALYSIS_OUTPUT_DIR / "dp_rmse_by_target_direction_sector.png", dpi=200, bbox_inches="tight"
)
plt.close(fig_hist)

fig_scatter, ax_scatter = plt.subplots(figsize=(7, 5))
ax_scatter.scatter(pred_df["target_dp_deg"], pred_df["dp_error_deg"], s=4, alpha=0.25)
ax_scatter.set_title("Direction error polar proxy")
ax_scatter.set_xlabel("Target Dp (deg)")
ax_scatter.set_ylabel("Dp error (deg)")
fig_scatter.tight_layout()
fig_scatter.savefig(ANALYSIS_OUTPUT_DIR / "dp_error_scatter.png", dpi=200, bbox_inches="tight")
plt.close(fig_scatter)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].bar(dir_sector_summary["target_dir_sector"].astype(str), dir_sector_summary["dir_rmse"])
axes[0].tick_params(axis="x", rotation=45)
axes[0].set_title("Dir RMSE by target direction sector")
axes[0].set_xlabel("Target Dir sector (deg)")
axes[0].set_ylabel("RMSE (deg)")
axes[1].scatter(pred_df["target_dir_deg"], pred_df["dir_error_deg"], s=4, alpha=0.25)
axes[1].set_title("Direction error polar proxy")
axes[1].set_xlabel("Target Dir (deg)")
axes[1].set_ylabel("Dir error (deg)")
plt.tight_layout()
plt.show()

fig_hist, ax_hist = plt.subplots(figsize=(7, 5))
ax_hist.bar(dir_sector_summary["target_dir_sector"].astype(str), dir_sector_summary["dir_rmse"])
ax_hist.tick_params(axis="x", rotation=45)
ax_hist.set_title("Dir RMSE by target direction sector")
ax_hist.set_xlabel("Target Dir sector (deg)")
ax_hist.set_ylabel("RMSE (deg)")
fig_hist.tight_layout()
fig_hist.savefig(
    ANALYSIS_OUTPUT_DIR / "dir_rmse_by_target_direction_sector.png", dpi=200, bbox_inches="tight"
)
plt.close(fig_hist)

fig_scatter, ax_scatter = plt.subplots(figsize=(7, 5))
ax_scatter.scatter(pred_df["target_dir_deg"], pred_df["dir_error_deg"], s=4, alpha=0.25)
ax_scatter.set_title("Direction error polar proxy")
ax_scatter.set_xlabel("Target Dir (deg)")
ax_scatter.set_ylabel("Dir error (deg)")
fig_scatter.tight_layout()
fig_scatter.savefig(ANALYSIS_OUTPUT_DIR / "dir_error_scatter.png", dpi=200, bbox_inches="tight")
plt.close(fig_scatter)